# 29장 · 오케스트레이터–워커 · 하위 질문을 계획하고 검증하기

오케스트레이터는 입력을 보고 하위 작업을 정합니다. 워커는 그중 작업 하나를 처리합니다. 모델이 작성한 계획을 파이썬에서 순회하려면 우선 응답 문자열을 목록으로 바꿔야 합니다.

## JSON 작업 목록에서 한 항목 읽기

`json.loads`는 JSON 형식의 문자열을 파이썬 값으로 바꿉니다. 아래 입력은 대괄호로 감싼 배열이므로 리스트가 됩니다. 변환한 뒤에야 `tasks[0]`이 첫 작업을 가리킵니다.

In [ ]:
import json

plan_text = '["제목 번역", "본문 요약"]'
tasks = json.loads(plan_text)
print(tasks[0])

제목 번역


출력은 첫 작업인 “제목 번역”입니다. 아래에서는 실제 모델이 계획을 쓰고, 항목 수와 원문 근거를 검사한 뒤 워커에 넘깁니다.

## 응용 실습 · 복합 문의를 하위 작업으로 나누기

## 쪼갤 단위 정하기

Q1의 “언제 어디로”는 시간과 장소를 묻지만 같은 반납 행동의 조건입니다. “친구가 대신 받을 수 있을까”는 수령 주체에 관한 별도 질문입니다. 둘로 나눈다면 수령 자격 하나, 반납 기한·장소·반납물 하나가 자연스럽습니다. 친구와 케이블처럼 눈에 띄는 명사만 남기면 대상은 남아도 물음이 사라집니다.

## 실습 준비

8장에서 모델을 호출했던 Jupyter 커널을 선택하고, 이 장의 실습 묶음을 풀어 노트북과 pre-mid 폴더를 나란히 둡니다. 새로운 커널이라면 빈 셀에서 `%pip install -U openai`를 실행하고 수업용 API 키를 환경 변수로 설정한 뒤 시작하세요. 키는 노트북 셀이나 제출물에 적지 않습니다. 아래 함수에는 매번 지시와 이번에 읽을 자료를 전달합니다. 출력은 실행할 때마다 표현이 달라질 수 있습니다.

In [ ]:
from pathlib import Path
import os
from openai import OpenAI

# ZIP을 푼 폴더에서 노트북을 열면 이 경로에 네 가지 대여 조항이 있습니다.
rules = Path("pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")
model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
question = '6월 10일에 A형을 빌리기로 했고 승인 댓글도 받았어요. 수업이 늦게 끝나는데 친구가 대신 받아도 될까요? 제가 직접 받으면 케이블은 언제 어디로 돌려주면 되나요?'
print(question)

## 계획의 evidence와 question 필드 정의하기

계획을 만드는 호출은 각 하위 작업을 두 필드로 반환합니다. `evidence`에는 원문에서 그대로 가져온 구절을, `question`에는 다음 호출이 독립적으로 답할 질문을 넣습니다. 아래 지시문은 전체 답을 작성하기 전에 이 배열만 받도록 합니다.

In [ ]:
import json

plan_instruction = """서로 다른 작업을 최대 2개로 묶으세요.
각 작업의 evidence는 문의에서 그대로 옮긴 연속 구절, question은 독립된 물음입니다.
독립적으로 답하도록 question에 필요한 날짜·승인 상태를 보존하세요.
응답은 [{"evidence":"원문 구절","question":"하위 질문"}] 형태의 JSON 배열만 쓰세요."""

In [ ]:
raw_plan = ask(plan_instruction, question)
print(raw_plan)

두 항목이 나왔는지만 세지 말고 각 근거 구절을 원문에서 찾으세요. “저라면 빨리 가겠습니다”처럼 문의에 없는 권고는 하위 질문이 아닙니다. 두 번째 항목에 반납할 물건, 시점, 장소가 한데 남았는지 확인합니다.

### JSON으로 읽어도 작업 목록인지 따로 검사하기

`json.loads`는 올바른 JSON 문자열을 파이썬 값으로 바꿉니다. JSON으로 읽을 수 있다고 해서 작업 목록을 받았다는 뜻일까요?

In [ ]:
import json

raw = '{"question": "몇 시인가요?"}'
parsed = json.loads(raw)
print(type(parsed).__name__, isinstance(parsed, list))

dict False


이 문자열은 올바른 JSON 객체여서 dict가 됩니다. 다음 함수가 기대하는 것은 작업 여러 개를 담은 list이므로, 이런 응답은 작업을 시작하기 전에 형식 검사에서 걸러집니다.

## read_plan으로 작업 개수와 원문 인용 검사하기

`read_plan(raw, original)`은 모델이 쓴 JSON 문자열과 원래 문의를 인자로 받습니다. 통과하면 작업 딕셔너리 목록이 돌아옵니다. 작업이 세 개라면 다음 호출을 시작하기 전에 `ValueError`가 납니다.

텍스트가 JSON처럼 보이는 것과 프로그램이 읽을 수 있는 것은 다릅니다. json.loads가 텍스트를 목록과 딕셔너리로 읽습니다. 형식, 개수, 원문 인용을 확인하고 통과한 항목만 다음 단계에 보냅니다. 상한을 넘긴 목록은 앞의 두 항목만 취하지 않고 이유와 함께 멈춥니다. 빠뜨린 세 번째 요구가 무엇인지 다시 읽을 기회를 보존합니다.

오케스트레이터가 세운 계획은 처음에는 JSON 모양의 문자열입니다. `json.loads(raw)`가 Python 값으로 바꾸지만 결과가 반드시 목록이라는 보장은 없습니다. `read_plan(raw, original)`이 항목 수와 각 질문의 근거를 검사하고 검증한 목록을 반환합니다. 형식이 올바른 `[]`도 이 수업에서는 실행할 작업이 0개라서 여기서 멈춥니다. 검사를 지난 항목만 다음 작업자 호출에 씁니다.

### 응답 문자열에서 하위 질문 하나 꺼내기

In [ ]:
import json

raw_item = '[{"evidence": "반납 장소", "question": "어디로 반납하나요?"}]'
parsed_items = json.loads(raw_item)
print(type(parsed_items).__name__)
print(parsed_items[0]["question"])

list
어디로 반납하나요?


첫 줄의 list는 이제 파이썬이 항목을 하나씩 읽을 수 있다는 뜻입니다. 아직 근거 구절이 원문에 있는지는 확인하지 않았습니다. 다음 함수는 원문도 인자로 받아 그 검사를 수행합니다.

In [ ]:
def read_plan(raw, original):
    parts = json.loads(raw)
    if not isinstance(parts, list) or not 1 <= len(parts) <= 2:
        raise ValueError("요구를 1~2개로 다시 묶으세요.")
    seen = set()
    for part in parts:
        if not isinstance(part, dict):
            raise ValueError("각 항목에는 evidence와 question이 필요합니다.")
        evidence, task = part.get("evidence"), part.get("question")
        if not isinstance(evidence, str) or not evidence.strip() or evidence not in original:
            raise ValueError("원문에서 근거 구절을 찾을 수 없습니다.")
        if not isinstance(task, str) or not task.strip() or task in seen:
            raise ValueError("하위 질문이 비었거나 중복됐습니다.")
        seen.add(task)
    return parts

plan = read_plan(raw_plan, question)
print([part["evidence"] for part in plan])

정확히 같은 문자열 중복은 잡지만 “언제 돌려줘요?”와 “반납 기한이 언제예요?”는 둘 다 통과할 수 있습니다. 형식 검사를 통과한 뒤 사람이 원문의 모든 요구를 덮는지 다시 읽어야 합니다. 검사에서 멈췄다면 원 응답을 확인하고 모델에 앞뒤 설명 없이 JSON만 쓰도록 한 번 더 요청하세요.

In [ ]:
answers = []
for part in plan:
    # 항목을 순서대로 처리하며 답과 출처가 어느 질문의 것인지 함께 보관합니다.
    reply = ask("규정으로 이 질문에 답하고 근거 조항을 붙이세요.", rules + "\n" + part["question"])
    answers.append({"질문": part["question"], "근거 구절": part["evidence"], "답": reply})
    print(part["question"], reply)

각 답이 독립적이어도 처음 정한 날짜를 잃으면 반납 시점 계산에 실패할 수 있습니다. 저장한 질문을 보며 필요한 날짜가 실제로 모델에 전달됐는지 찾아보세요. 다음 장에서는 이 부분 답을 다시 한 사람에게 보낼 안내로 엮습니다.

## 질문의 개수와 빠짐을 따로 판정하기

**사람이 작성한 계획 예시**

| 원문 구절 | 독립적으로 읽을 하위 질문 | 답의 근거 |
|---|---|---|
| “친구가 대신 받아도 될까요?” | 6월 10일 A형 대여 승인을 받은 회원의 친구가 대신 받을 수 있나요? | R2 직접 수령 |
| “케이블은 언제 어디로 돌려주면 되나요?” | 6월 10일 직접 수령하면 기기와 케이블을 언제 어디로 반납하나요? | R3 다음 날 오후 5시, 가온관 1층 안내 데스크 |

“친구가 대신 받아도 될까요?”와 “제가 직접 가야 하나요?”만 있으면 두 항목이어도 반납 문의는 비어 있습니다.

## 무엇을 하나의 하위 질문으로 묶을까

두 사람이 회원 문의를 번갈아 읽습니다. 첫 사람은 명사마다 질문을 하나 만들고, 둘째 사람은 회원이 해야 할 행동마다 하나를 만듭니다. 전자는 “케이블”, “장소”, “날짜” 세 장이 될 수 있습니다. 어느 날짜가 수령일이고 어느 장소가 반납 장소인지 카드에 남나요? 행동으로 묶은 카드도 수령 담당자에게 반납 기한을 묻는 식으로 섞일 수 있습니다. 카드 한 장만 받은 사람이 원래 대상을 정확히 찾을 수 있는지 서로 교환해서 확인합니다.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 실습 · 세 조건을 두 작업에 배치하기

개인 실습 90분. “A형 수령 승인을 받았어요. 오늘 몇 시 어디서 받나요? 직접 반납할 때 케이블도 가져가야 하나요?”를 보냅니다. 생성 전에 수령 시각·장소와 반납물을 어느 항목에 넣을지 정합니다. 같은 물음을 반복한 계획과 항목 셋인 계획을 직접 만들어 판정해 보세요. 프로그램이 잡은 문제와 사람이 찾아야 했던 문제를 구분합니다.

In [ ]:
new_question = "A형 수령 승인을 받았어요. 오늘 몇 시 어디서 받나요? 직접 반납할 때 케이블도 가져가야 하나요?"
# 두 항목에 원문의 세 조건이 모두 남는지 확인합니다.
# ask가 돌려준 JSON 문자열과 new_question을 read_plan에 넘기세요.
# 각 작업의 evidence와 question을 짝지어 읽고 시각·장소·케이블이 남아 있는지 표시하세요.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>new_question = &quot;A형 수령 승인을 받았어요. 오늘 몇 시 어디서 받나요? 직접 반납할 때 케이블도 가져가야 하나요?&quot;
parts = read_plan(ask(plan_instruction, new_question), new_question)
for part in parts:
    print(part[&quot;evidence&quot;], part[&quot;question&quot;])
manual = {&quot;evidence&quot;: &quot;오늘 몇 시 어디서 받나요?&quot;, &quot;question&quot;: &quot;수령 시각은 언제인가요?&quot;}
for proposed in [[manual, manual], [manual, manual, manual],
                 [manual, {**manual, &quot;question&quot;: &quot;몇 시에 수령해요?&quot;}]]:
    try:
        print(&quot;형식 검사 통과&quot;, read_plan(json.dumps(proposed, ensure_ascii=False), new_question))
    except ValueError as error:
        print(&quot;다시 계획할 이유&quot;, error)</code></pre>
<p>수령 시각과 장소를 R2에 함께 대응시키고, 케이블 반납을 R3에 대응할 수 있습니다. 세 항목은 상한 검사에서 멈춥니다. 의미가 같은 두 문장을 다른 말로 썼다면 인용과 형식이 모두 정상일 수 있으므로 원문 체크가 남습니다.</p>
</details>

## 다음 작업에 가져갈 기록

하위 질문, 근거 구절, 규정 조항, 답을 짝지은 작업 기록을 보관하세요. 다음 장의 취합은 원문의 요구와 이 작업 기록을 함께 받습니다.

[OpenAI Python SDK 사용 예](https://github.com/openai/openai-python)